In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep
import os
import random

In [2]:
# install on the control webiste librbay code. 
# import subprocess
# import sys

# required_packages = ['python-docx',  'pypandoc']

# for package in required_packages:
#     try:
#         __import__(package)
#     except ImportError:
#         print(f'Installing {package}...')
#         subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

In [3]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'II_CAN BCFSA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running II_CAN BCFSA Web Scraping Tool v.1.1


In [4]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

In [5]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

         regulatorName + ' 1': 'https://www.bcfsa.ca/public-resources/credit-unions/bc-authorized-credit-unions/find-credit-union-search-results?type=credit_union',
         regulatorName + ' 2': 'https://www.bcfsa.ca/public-resources/insurance-business/authorized-insurance-companies/authorized-insurance-company-search-results?type=insurance',
         regulatorName + ' 3': 'https://www.bcfsa.ca/public-resources/trust-business/regulated-trust-companies/regulated-trust-company-search-results?type=trust',
         regulatorName + ' 4': 'https://www.bcfsa.ca/public-resources/credit-union-deposit-insurance/bc-credit-unions/find-cudic-authorized-credit-union-search-results?type=cudic',

        }



Typology={

       regulatorName + ' 1': 'List of Credit Unions',
       regulatorName + ' 2': 'List of Authorized Insurance Companies',
       regulatorName + ' 3': 'List of Trust Companies',
       regulatorName + ' 4': 'List of CUDIC Authorized Credit Union',


        }

In [6]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


In [7]:
#------------------------------------------------ Begin_Main ----------------------------------------

# Process each link with a new browser instance to prevent security blocking
for reg, url in regdict.items():
    list_name = Typology.get(reg, 'Unknown List')
    print('Working with {} - {}'.format(reg, list_name))
    
    # Open a new browser instance for this link
    driver = webdriver.Chrome(options=chromeOptions)
    driver.maximize_window()
    
    driver.get(url)
    sleep(random.uniform(4, 8)) # random sleep to prevent security block

    # Continue scraping while there is a next page
    while True:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find("table")
        if not table:
            print("No table element found for {}".format(reg))
        else:
            tbody = table.find("tbody")
            if not tbody:
                print("No <tbody> found in the table for {}".format(reg))
            else:
                rows = tbody.find_all("tr")
                rows_count = len(rows)
                print(f"[INFO] : Found {rows_count} rows on {reg}")
                for row in rows:
                    tds = row.find_all("td")
                    name_val = tds[0].get_text(strip=True)
                    business = tds[1].get_text(strip=True)
                    address_val = tds[2].get_text(" ", strip=True)
                    website = tds[3].get_text(strip=True)
                    
                    sqldict['Name'].append(name_val)
                    sqldict['Address_1'].append(address_val)
                    sqldict['Website'].append(website)
                    sqldict['CoType'].append(business)
                        
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append('II_CAN')
                    sqldict['RegCode'].append('BCFSA')
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['RegulationType'].append("Regulated")
                    sqldict['ListName'].append(list_name)

                    sqldict = bourange_same_length_array(sqldict)
        
        # Check for the next button
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, "li.pager__item--next a.pager__link")
        except Exception as e:
            print("[INFO] : Next button not found for {}, continuing to next link.".format(reg))
            break  # Exit pagination for this link
        
        # If the next button is disabled, exit pagination
        if next_btn.get_attribute("disabled") is not None:
            print("[INFO] : Next button is disabled for {}, continuing to next link.".format(reg))
            break
        else:
            # Instead of clicking, retrieve its URL and open it in a new instance to prevent security block
            next_url = next_btn.get_attribute("href")
            if not next_url:
                print("[INFO] : Next URL not found, continuing to next link for {}.".format(reg))
                break
            
            print("[INFO] : Pagination - loading next page for {}: {}".format(reg, next_url))
            driver.quit()  # Close the current instance
            sleep(random.uniform(4, 8))
            driver = webdriver.Chrome(options=chromeOptions)
            driver.maximize_window()
            driver.get(next_url)
            wait = WebDriverWait(driver, 20)
            wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            sleep(random.uniform(4, 8))
    
    driver.quit()  
    print(f"[INFO] : Completed {reg}")
    sleep(random.uniform(4, 8))

Working with II_CAN BCFSA 3 - List of Trust Companies
[INFO] : Found 50 rows on II_CAN BCFSA 3
[INFO] : Pagination - loading next page for II_CAN BCFSA 3: https://www.bcfsa.ca/public-resources/trust-business/regulated-trust-companies/regulated-trust-company-search-results?type=trust&page=2
[INFO] : Found 4 rows on II_CAN BCFSA 3
[INFO] : Next button is disabled for II_CAN BCFSA 3, continuing to next link.
[INFO] : Completed II_CAN BCFSA 3


In [8]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
driver.quit()
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully")

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_2688\74031927.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


[INFO] : Excel file 'II_CAN BCFSA SQL Ready 2026-04-15 14.37.58.xlsx' saved successfully


In [9]:
#------------------------------------------------ Data Integrity & Consistency Check ----------------------------------------

print("="*80)
print("DATA INTEGRITY & CONSISTENCY VERIFICATION")
print("="*80)

# Expected values from README_II_CAN_BCFSA.md
expected_lists = {
    '1': {'name': 'List of Credit Unions'},
    '2': {'name': 'List of Authorized Insurance Companies'},
    '3': {'name': 'List of Trust Companies'},
    '4': {'name': 'List of CUDIC Authorized Credit Union'}
}

print("\n1. DATAFRAME SHAPE:")
print(f"   Total rows collected: {len(df)}")
print(f"   Total columns: {len(df.columns)}")

print("\n2. DATA DISTRIBUTION BY LIST:")
list_summary = df.groupby('ListCode').agg({
    'Name': 'count',
    'ListName': 'first',
    'RegCtry': 'first',
    'RegCode': 'first'
}).rename(columns={'Name': 'Count'})
print(list_summary)

print("\n3. NULL VALUES CHECK (Data Completeness):")
null_counts = df.isnull().sum()
if null_counts.sum() == 0:
    print("   ✓ PASS: No null values found")
else:
    print("   ✗ FAIL: Null values detected in:")
    for col, count in null_counts[null_counts > 0].items():
        print(f"      - {col}: {count} nulls")

print("\n4. CONSISTENCY WITH README:")
print("\n   List | Expected Name                          | Actual Name                         | Count | Status")
print("   -----|----------------------------------------|-------------------------------------|-------|--------")
for list_code in ['1', '2', '3', '4']:
    data = df[df['ListCode'] == list_code]
    if len(data) == 0:
        print(f"   {list_code}    | {expected_lists[list_code]['name']:<37} | (NO DATA)                           |   0   | ✗ MISSING")
    else:
        actual_name = data['ListName'].iloc[0]
        expected_name = expected_lists[list_code]['name']
        count = len(data)
        match = "✓ OK" if expected_name.lower() in actual_name.lower() else "✗ MISMATCH"
        print(f"   {list_code}    | {expected_name:<37} | {actual_name:<34} | {count:>5} | {match}")

print("\n5. REGCTRY & REGCODE VALIDATION:")
regctry_values = df['RegCtry'].unique()
regcode_values = df['RegCode'].unique()
regctry_status = "✓ CORRECT" if all(v == 'II_CAN' for v in regctry_values) else "✗ INCORRECT"
regcode_status = "✓ CORRECT" if all(v == 'BCFSA' for v in regcode_values) else "✗ INCORRECT"
print(f"   RegCtry values: {regctry_values} {regctry_status}")
print(f"   RegCode values: {regcode_values} {regcode_status}")

print("\n6. KEY FIELDS VALIDATION:")
name_check = (df['Name'].notna().sum(), len(df), "✓ OK" if df['Name'].notna().all() else "✗ INCOMPLETE")
website_check = (df['Website'].notna().sum(), len(df), "✓ OK" if df['Website'].notna().all() else "✗ INCOMPLETE")
cotype_check = (df['CoType'].notna().sum(), len(df), "✓ OK" if df['CoType'].notna().all() else "✗ INCOMPLETE")
address_check = (df['Address_1'].notna().sum(), len(df), "✓ OK" if df['Address_1'].notna().all() else "✗ INCOMPLETE")

print(f"   Name field filled: {name_check[0]}/{name_check[1]} {name_check[2]}")
print(f"   Website field filled: {website_check[0]}/{website_check[1]} {website_check[2]}")
print(f"   CoType field filled: {cotype_check[0]}/{cotype_check[1]} {cotype_check[2]}")
print(f"   Address_1 field filled: {address_check[0]}/{address_check[1]} {address_check[2]}")
print(f"   ListProcessDate: {df['ListProcessDate'].iloc[0] if len(df) > 0 else 'N/A'}")

print("\n7. SAMPLE DATA (first 3 rows):")
if len(df) > 0:
    print(df[['Name', 'CoType', 'Address_1', 'Website', 'ListCode', 'ListName']].head(3).to_string())
else:
    print("   NO DATA COLLECTED")

print("\n8. COMPARISON WITH README REQUIREMENTS:")
print("   List | Expected Name                          | Collected Rows | Status")
print("   ----|----------------------------------------|----------------|--------")
total_expected = 4
total_collected = 0
for list_code, expected in expected_lists.items():
    count = len(df[df['ListCode'] == list_code])
    total_collected += (1 if count > 0 else 0)
    if count == 0:
        status = "✗ MISSING"
    else:
        status = "✓ COLLECTED"
    print(f"   {list_code}   | {expected['name']:<37} | {count:>14} | {status}")

print(f"\n   Lists collected: {total_collected}/{total_expected}")

print("\n" + "="*80)
print("SUMMARY:")
print("="*80)
print(f"✓ Total rows in DataFrame: {len(df)}")
print(f"✓ Expected lists coverage: {total_collected}/{total_expected}")
missing_lists = total_expected - total_collected
if missing_lists > 0:
    print(f"✗ Missing lists count: {missing_lists} (Lists 2 & 4 likely missing due to table structure differences)")
    print(f"  ACTION: Need to inspect HTML structure of missing lists and update CSS selectors")
else:
    print(f"✓ All lists collected successfully")

print("="*80)

DATA INTEGRITY & CONSISTENCY VERIFICATION

1. DATAFRAME SHAPE:
   Total rows collected: 54
   Total columns: 44

2. DATA DISTRIBUTION BY LIST:
          Count                 ListName RegCtry RegCode
ListCode                                                
3            54  List of Trust Companies  II_CAN   BCFSA

3. NULL VALUES CHECK (Data Completeness):
   ✓ PASS: No null values found

4. CONSISTENCY WITH README:

   List | Expected Name                          | Actual Name                         | Count | Status
   -----|----------------------------------------|-------------------------------------|-------|--------
   1    | List of Credit Unions                 | (NO DATA)                           |   0   | ✗ MISSING
   2    | List of Authorized Insurance Companies | (NO DATA)                           |   0   | ✗ MISSING
   3    | List of Trust Companies               | List of Trust Companies            |    54 | ✓ OK
   4    | List of CUDIC Authorized Credit Union | (NO DATA)